# Lasso Regression: Predicting Diabetes

**Goal:** predict whether a person has diabetes (`Diabetes_binary`: 0 = no, 1 = yes) from health and lifestyle answers in a 2015 CDC survey.

This notebook uses a **lasso model**: logistic regression with an extra rule that lets the model decide which columns matter.

1. **Split** the data into a training set and a test set
2. **Build** the model inputs (`X`) and the answer (`y`), using **all** the columns this time
3. **Set class weights** so the model doesn't ignore the smaller group
4. **Scale** the columns so they are on a similar size
5. **Train** the lasso model
6. **See** which columns lasso kept
7. **Predict** on the test set
8. **Evaluate** with accuracy, precision, and recall

## Load the data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv("diabetes_binary_health_indicators_BRFSS2015.csv")

# List of variables that are logically binary/categorical (factor in R)
binary_vars = [
    "Diabetes_binary", "HighBP", "HighChol", "CholCheck", "Smoker", "Stroke",
    "HeartDiseaseorAttack", "PhysActivity", "Fruits", "Veggies",
    "HvyAlcoholConsump", "AnyHealthcare", "NoDocbcCost", "DiffWalk", "Sex",
]

# Convert those columns to pandas 'category' dtype (equivalent of as.factor)
for col in binary_vars:
    df[col] = df[col].astype("category")

print(df.head())
print(df.dtypes)

## Step 1: Split into training and test sets

A model should be judged on data it has **never seen**. 

So we split the data:
- **Training set (80%)**: the model learns from this.
- **Test set (20%)**: we hide it and use it at the end to check the model.

In [ ]:
diabetic_train, diabetic_test = train_test_split(
    df, test_size=0.2, stratify=df["Diabetes_binary"], random_state=1
)

## Step 2: Build the model inputs (`X`) and the answer (`y`)

- **`X`**: the columns the model uses to make a prediction (the *predictors*)
- **`y`**: the answer we want it to predict

For the lasso model we use **all 21 columns**, not just 10. It can handle lots of columns and shrink the ones that don't help.

In [ ]:
# Build the design (model) matrix for the chosen predictors
# Lasso gets ALL the columns (every column except the answer) and decides which ones matter
predictors = [col for col in df.columns if col != "Diabetes_binary"]

X_train = pd.get_dummies(diabetic_train[predictors], drop_first=True)
y_train = diabetic_train["Diabetes_binary"].astype(int)

X_test = pd.get_dummies(diabetic_test[predictors], drop_first=True)
# Make sure the test columns match the training columns exactly
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
y_test = diabetic_test["Diabetes_binary"].astype(int)

## Step 3: Class weights

Most people in the survey do **not** have diabetes. If the model sees mostly "no" answers, it can score well by almost always guessing "no" while missing the people we care about most.

**Class weights** fix this: each person in the smaller group counts for more during training, so both groups matter equally.

In [ ]:
class_counts = y_train.value_counts()
inv_weights = 1 / class_counts
sample_weights = y_train.map(inv_weights)

## Step 4: Put the columns on the same scale

`BMI` can be 40, while a yes/no column is only 0 or 1. Lasso's penalty works best when all columns are on a similar size, so we **scale** them: each column is shifted and stretched so its average is 0 and its typical spread is about 1.

We learn the scale from the **training data only**, then apply it to both sets. That keeps the test set truly hidden.

`fit` learns each column's average and standard deviation from the training data.

`transform` uses those numbers to scale the values.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler() # adjusts each coloumn so its average is approximately 0 and its standard deviation is 1

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test), columns=X_train.columns, index=X_test.index
)

X_train_scaled.head()

- **0** = equal to the average
- **Positive** = above average
- **Negative** = below average
- **1 or -1** = one standard devaition above or below average.

## Step 5: Train the lasso model

**Lasso** is logistic regression with one extra rule: the model is *penalized* for relying on columns that don't help. The penalty shrinks weak columns toward zero, and sometimes all the way to exactly **0**, which means the model ignores that column.

The penalty has a strength setting. `LogisticRegressionCV` tries 10 different strengths and uses **5-fold cross-validation** (it splits the training data 5 ways, trains on some parts, and checks on the rest) to pick the best one. The test set is never touched. This step can take a minute or two.

As before, we scale the class weights up so their average is 1. The smaller group still counts more.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)   # hides a harmless version warning

from sklearn.linear_model import LogisticRegressionCV
scaled_weights = sample_weights * len(y_train) / 2

model = LogisticRegressionCV(
    Cs=10,               # try 10 penalty strengths
    cv=5,                # 5-fold cross-validation
    penalty="l1",        # "l1" means lasso
    solver="liblinear",
    scoring="roc_auc",   # pick the strength that ranks people best
    max_iter=1000,
    random_state=1,
)
model.fit(X_train_scaled, y_train, sample_weight=scaled_weights)

print("Model trained!")
print("Best penalty strength (C):", model.C_[0], "(smaller C = stronger penalty)")

## Step 6: Which columns did lasso keep?

Each column gets a **coefficient**:
- **Positive**: higher values push the probability of diabetes **up**
- **Negative**: higher values push it **down**
- **Exactly 0**: lasso dropped the column

Because we scaled the columns, longer bars mean a bigger effect.

In [ ]:
coefs = pd.Series(model.coef_[0], index=X_train.columns).sort_values()

dropped = list(coefs[coefs == 0].index)
print("Columns lasso dropped (coefficient = 0):", dropped if dropped else "none, lasso kept every column")
print("Columns kept:", len(coefs) - len(dropped), "out of", len(coefs))

coefs.plot(kind="barh", figsize=(8, 7), title="Lasso coefficients")
plt.axvline(0, color="black", linewidth=0.8)
plt.show()

If lasso dropped some columns, those didn't add enough to be worth keeping. If it dropped none, the penalty at the best strength was mild and every column helped a little.

## Step 7: Make predictions

The model gives a **probability** for each person in the test set. We turn that into a yes/no prediction using a **cutoff**: 0.55 or higher means we predict "diabetes".

We use the same cutoff as the logistic regression notebook so the models can be compared fairly.

In [ ]:
pred_probs = model.predict_proba(X_test_scaled)[:, 1]   # probability of diabetes
pred_class = (pred_probs >= 0.55).astype(int)          # 1 = diabetes, 0 = no diabetes

# Look at the first 10 predictions next to the real answers
pd.DataFrame({
    "probability": pred_probs[:10].round(3),
    "predicted": pred_class[:10],
    "actual": y_test.values[:10],
})

## Step 8: Accuracy vs. Precision vs. Recall

- **Accuracy**: out of everyone, how many did the model get right?
- **Precision**: of the people the model flagged as having diabetes, how many really do?
- **Recall**: of the people who really have diabetes, how many did the model catch?

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

accuracy = accuracy_score(y_test, pred_class)
precision = precision_score(y_test, pred_class)
recall = recall_score(y_test, pred_class)

print("Accuracy vs. Precision vs. Recall")
print(f"  Accuracy : {accuracy:.3f}")
print(f"  Precision: {precision:.3f}")
print(f"  Recall   : {recall:.3f}")

**How to read these:** accuracy alone can look fine even if the model misses many people with diabetes, so check **recall** too. Precision tells you how many of the model's "diabetes" flags are correct.

**Compare:** in the logistic regression notebook (10 columns, same split and cutoff) the scores were Accuracy 0.748, Precision 0.317, Recall 0.698. Did the extra columns help the lasso model?

## Recap

| Step | What we did | Why |
|------|-------------|-----|
| 1 | Split into training (80%) and test (20%) | Judge the model on data it hasn't seen |
| 2 | Built `X` and `y` with all 21 columns | Give the model lots of information |
| 3 | Made class weights | Keep the model from ignoring the smaller group |
| 4 | Scaled the columns | Make the lasso penalty fair across columns |
| 5 | Trained a lasso model | Shrink columns that don't help |
| 6 | Looked at the coefficients | See which columns matter and which were dropped |
| 7 | Predicted probabilities, then yes/no at a 0.55 cutoff | Turn the output into answers we can check |
| 8 | Printed accuracy, precision, and recall | See how well the model does on the test set |